# 02 — Train and Evaluate

**Main notebook.** Trains ResNet-50 and EfficientNet-B0 on HAM10000 with transfer learning.

Handles class imbalance via weighted loss. Compares both models side by side.

**Estimated runtime:** ~1-2 hours on Colab T4 GPU.

In [ ]:
# --- Environment Setup ---
import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !pip install -q torch torchvision pandas scikit-learn matplotlib seaborn Pillow tqdm

    # Clone the repo to get src/ modules
    !git clone https://github.com/daniel-duhnev/advanced-topics-upf.git /content/advanced-topics 2>/dev/null || true
    os.chdir('/content/advanced-topics')
    sys.path.insert(0, '/content/advanced-topics')

    # Download dataset from Kaggle
    !pip install -q kaggle
    from google.colab import files
    # Upload kaggle.json when prompted:
    if not os.path.exists(os.path.expanduser('~/.kaggle/kaggle.json')):
        print("Upload your kaggle.json file:")
        uploaded = files.upload()
        !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

    !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p /content/data/ --quiet
    !unzip -q -o /content/data/skin-cancer-mnist-ham10000.zip -d /content/data/HAM10000/
    DATA_DIR = '/content/data/HAM10000'
else:
    # Local execution (Mac with MPS or Linux with CUDA)
    DATA_DIR = '../data/HAM10000'
    sys.path.insert(0, '..')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'MPS available: {torch.backends.mps.is_available()}')
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print(f'Using device: {device}')

In [ ]:
# --- Reproducibility ---
import numpy as np
import random

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

In [ ]:
# --- Load Data ---
from pathlib import Path
from src.dataset import get_dataloaders, get_transforms
from src.config import DEVICE

metadata_path = Path(DATA_DIR) / 'HAM10000_metadata.csv'

train_loader, val_loader, test_loader, class_weights = get_dataloaders(metadata_path, Path(DATA_DIR))

print(f'Train batches: {len(train_loader)} ({len(train_loader.dataset)} images)')
print(f'Val batches:   {len(val_loader)} ({len(val_loader.dataset)} images)')
print(f'Test batches:  {len(test_loader)} ({len(test_loader.dataset)} images)')
print(f'\nClass weights: {class_weights}')
print(f'Device: {DEVICE}')

In [ ]:
# --- Visualize Augmentations ---
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

# Show a batch of augmented training images
images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i, ax in enumerate(axes.flat):
    if i < len(images):
        img = images[i].permute(1, 2, 0).numpy()
        img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])  # Denormalize
        img = np.clip(img, 0, 1)
        ax.imshow(img)
    ax.axis('off')
plt.suptitle('Augmented Training Samples')
plt.tight_layout()
plt.show()

## Train ResNet-50

In [ ]:
from src.models import get_resnet50
from src.train import train_model

resnet = get_resnet50(num_classes=7, pretrained=True)
trainable = sum(p.numel() for p in resnet.parameters() if p.requires_grad)
total = sum(p.numel() for p in resnet.parameters())
print(f'ResNet-50: {trainable:,} trainable / {total:,} total parameters')

resnet, resnet_history = train_model(resnet, train_loader, val_loader, class_weights, model_name='resnet50')

## Train EfficientNet-B0

In [ ]:
from src.models import get_efficientnet_b0

effnet = get_efficientnet_b0(num_classes=7, pretrained=True)
trainable = sum(p.numel() for p in effnet.parameters() if p.requires_grad)
total = sum(p.numel() for p in effnet.parameters())
print(f'EfficientNet-B0: {trainable:,} trainable / {total:,} total parameters')

effnet, effnet_history = train_model(effnet, train_loader, val_loader, class_weights, model_name='efficientnet_b0')

## Evaluate Both Models on Test Set

In [ ]:
from src.evaluate import evaluate_model, compute_metrics, plot_confusion_matrix, plot_training_curves
from src.config import MODELS_DIR

# Load best checkpoints
resnet.load_state_dict(torch.load(MODELS_DIR / 'resnet50_best.pth', map_location=DEVICE))
effnet.load_state_dict(torch.load(MODELS_DIR / 'efficientnet_b0_best.pth', map_location=DEVICE))

print('=== ResNet-50 ===')
resnet_preds, resnet_labels = evaluate_model(resnet, test_loader)
resnet_metrics = compute_metrics(resnet_labels, resnet_preds)

print('\n=== EfficientNet-B0 ===')
effnet_preds, effnet_labels = evaluate_model(effnet, test_loader)
effnet_metrics = compute_metrics(effnet_labels, effnet_preds)

In [ ]:
# --- Confusion Matrices Side by Side ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_confusion_matrix(resnet_labels, resnet_preds, title='ResNet-50')
plot_confusion_matrix(effnet_labels, effnet_preds, title='EfficientNet-B0')

In [ ]:
# --- Training Curves ---
plot_training_curves(resnet_history, title='ResNet-50 Training Curves')
plot_training_curves(effnet_history, title='EfficientNet-B0 Training Curves')

In [ ]:
# --- Final Comparison Table ---
import pandas as pd

comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'F1 (macro)', 'F1 (weighted)'],
    'ResNet-50': [resnet_metrics['accuracy'], resnet_metrics['f1_macro'], resnet_metrics['f1_weighted']],
    'EfficientNet-B0': [effnet_metrics['accuracy'], effnet_metrics['f1_macro'], effnet_metrics['f1_weighted']],
})
comparison = comparison.round(4)
print('\n=== Model Comparison ===')
print(comparison.to_string(index=False))